## Self Attention

In [1]:
import torch

In [3]:
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],
        [0.55, 0.87, 0.66],
        [0.57, 0.85, 0.64],
        [0.22, 0.58, 0.33],
        [0.77, 0.25, 0.1],
        [0.05, 0.8, 0.55],
    ]
)

In [5]:
inputs.size()  # 6 words, embedding dimension = 3

torch.Size([6, 3])

In [8]:
query = inputs[1, ...]  # Select the word at index 1
query

tensor([0.5500, 0.8700, 0.6600])

In [10]:
# create empty zero matrix for attention weights to be calculated
attn_scores_2 = torch.empty(
    inputs.shape[0]
)  # For each token in the input sequence, we are going to get a weight
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(
        x_i, query
    )  # Dot product of the "unprojected" inputs and the "unprojected" query

In [17]:
print(
    attn_scores_2
)  # Notice how the largest value turns out to be the vector itself. Dot product is higher when vectors are similar

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [18]:
# Get a probability simplex, i.e. a vector where values are between 0-1 and they sum up to 1
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print(f"Attention weights: {attn_weights_2_tmp}")
print(f"Sum: {attn_weights_2_tmp.sum()}")

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: 1.0000001192092896


In [19]:
# But it is better to use softmax
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)


attn_weights_2_naive = softmax_naive(attn_scores_2)
print(f"Attention weights: {attn_weights_2_naive}")
print(f"Sum: {attn_weights_2_naive.sum()}")

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: 1.0


In [21]:
# Or more numerically stable, just use torch's softmax
attn_weights_2 = torch.softmax(attn_scores_2, 0)
print(f"Attention weights: {attn_weights_2}")
print(f"Sum: {attn_weights_2.sum()}")

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: 1.0


In [ ]:
# Now it is time to calculate the context vector
query = inputs[1, ...]
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i
context_vec_2

tensor([0.4419, 0.6515, 0.5683])

In [24]:
# Of course, we don't do this with for loops
print(f"Shape of the attention weights: {attn_weights_2.size()}")
print(f"Shape of the input sequence: {inputs.size()}")

Shape of the attention weights: torch.Size([6])
Shape of the input sequence: torch.Size([6, 3])


In [32]:
attn_weights_2

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [47]:
attn_weights = torch.softmax(inputs @ inputs.T, dim=1)
# Attention weights for all (in the end, we have 6x6, with dim=1, we iterate over dim 1)
# e.g. consider if it was 2, 3 -> [[1,2,3], [4,5,6]] here dim=1 means first fix the row, i.e. tensor[0, ...] then apply it over the row here [1,2,3]
# and in parallel it is also applied to tensor[1, ...] etc.
# but for it to generalize when we switch to batches, we can do dim=-1
attn_weights = torch.softmax(inputs @ inputs.T, dim=-1)
attn_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [46]:
context_vectors = attn_weights @ inputs
context_vectors

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

## Self-Attention with Trainable Weights

In [48]:
x_2 = inputs[1]  # demonstrate with inputs[1] again
d_in = inputs.shape[1]  # dimension of the input vectors
d_out = 2  # we want 2-dim context vectors

In [49]:
torch.manual_seed(123)

In [50]:
# Create matrices to project input vectors into query, key, value
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [52]:
# Well, now project the x_2
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)

tensor([0.4306, 1.4551])


In [55]:
# Now do this for all of the inputs
keys = inputs @ W_key  # (6, 3)x(3, 2) -> (6, 2)
values = inputs @ W_value
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


In [59]:
# Attention score for key 2 and query 2 (key-query dot product)
keys_2 = keys[1, ...]
attn_score_22 = keys_2.dot(query_2)
print(attn_score_22)

tensor(1.8524)


In [62]:
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [64]:
# This uses a different normalization
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / (d_k**0.5), dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [65]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


In [68]:
# Turn this into a class
from torch import nn


class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key  # -> projection to keys, each row a key
        values = x @ self.W_value  # -> projection to values, each row a value
        queries = x @ self.W_query  # -> projection to queries, each row a query

        attn_scores = (
            queries @ keys.T
        )  # -> queries are projected by the keys, but first transpose the keys so each col is a key. Then you get q1k1, q2k2, ...
        attn_weights = torch.softmax(attn_scores / (keys.shape[-1] ** 0.5), dim=-1)
        context_vec = (
            attn_weights @ values
        )  # -> rows of the context_vec is a linear combination of the rows of values, where weights are from attn_weights
        return context_vec

In [69]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [70]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / (keys.shape[-1] ** 0.5), dim=-1)
        context_vector = attn_weights @ values
        return context_vector

In [71]:
torch.manual_seed(789)

sa_v2 = SelfAttention_v2(d_in, d_out)

print(sa_v2(inputs))

tensor([[ 0.0293, -0.3242],
        [ 0.0284, -0.3250],
        [ 0.0284, -0.3251],
        [ 0.0271, -0.3272],
        [ 0.0267, -0.3278],
        [ 0.0277, -0.3262]], grad_fn=<MmBackward0>)


In [75]:
sa_v2.W_query.weight

Parameter containing:
tensor([[ 0.3161,  0.4568,  0.5118],
        [-0.1683, -0.3379, -0.0918]], requires_grad=True)

In [79]:
sa_v1.W_query = torch.nn.Parameter(sa_v2.W_query.weight.T)
sa_v1.W_value = torch.nn.Parameter(sa_v2.W_value.weight.T)
sa_v1.W_key = torch.nn.Parameter(sa_v2.W_key.weight.T)

In [85]:
torch.stack([inputs, inputs]).shape

torch.Size([2, 6, 3])

## Causal (Masked Attention)

In [87]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
print(attn_weights)

tensor([[0.1667, 0.1635, 0.1641, 0.1672, 0.1767, 0.1618],
        [0.1675, 0.1622, 0.1626, 0.1688, 0.1746, 0.1642],
        [0.1675, 0.1623, 0.1627, 0.1687, 0.1745, 0.1643],
        [0.1672, 0.1642, 0.1644, 0.1680, 0.1705, 0.1657],
        [0.1672, 0.1646, 0.1648, 0.1678, 0.1694, 0.1661],
        [0.1673, 0.1634, 0.1637, 0.1682, 0.1723, 0.1650]],
       grad_fn=<SoftmaxBackward0>)


In [88]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [90]:
masked_simple = attn_weights * mask_simple
print(masked_simple)

tensor([[0.1667, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1675, 0.1622, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1675, 0.1623, 0.1627, 0.0000, 0.0000, 0.0000],
        [0.1672, 0.1642, 0.1644, 0.1680, 0.0000, 0.0000],
        [0.1672, 0.1646, 0.1648, 0.1678, 0.1694, 0.0000],
        [0.1673, 0.1634, 0.1637, 0.1682, 0.1723, 0.1650]],
       grad_fn=<MulBackward0>)


In [92]:
row_sums = torch.sum(masked_simple, dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5080, 0.4920, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3401, 0.3295, 0.3303, 0.0000, 0.0000, 0.0000],
        [0.2519, 0.2474, 0.2477, 0.2530, 0.0000, 0.0000],
        [0.2005, 0.1975, 0.1976, 0.2012, 0.2032, 0.0000],
        [0.1673, 0.1634, 0.1637, 0.1682, 0.1723, 0.1650]],
       grad_fn=<DivBackward0>)


In [97]:
mask = torch.triu(
    torch.ones(context_length, context_length), diagonal=1
)  # triu = upper-triangular, diagonal=1 removes the diagonals (shifts the diagonal up by 1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
masked

tensor([[-0.0634,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.0921, -0.1375,    -inf,    -inf,    -inf,    -inf],
        [-0.0908, -0.1357, -0.1322,    -inf,    -inf,    -inf],
        [-0.0514, -0.0774, -0.0757, -0.0452,    -inf,    -inf],
        [-0.0420, -0.0635, -0.0623, -0.0367, -0.0229,    -inf],
        [-0.0672, -0.1005, -0.0980, -0.0596, -0.0254, -0.0873]],
       grad_fn=<MaskedFillBackward0>)

In [99]:
attn_weights = torch.softmax(
    masked / keys.shape[-1] ** 0.5, dim=-1
)  # computational trick
attn_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5080, 0.4920, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3401, 0.3295, 0.3303, 0.0000, 0.0000, 0.0000],
        [0.2519, 0.2474, 0.2477, 0.2530, 0.0000, 0.0000],
        [0.2005, 0.1975, 0.1976, 0.2012, 0.2032, 0.0000],
        [0.1673, 0.1634, 0.1637, 0.1682, 0.1723, 0.1650]],
       grad_fn=<SoftmaxBackward0>)

#### Adding Dropout

In [ ]:
torch.manual_seed(123)
dropout = nn.Dropout(0.5)  # Drop samples with prob 0.5
example = torch.ones(6, 6)
dropout(
    example
)  # will be scaled (half gone -> x2, this is for inference when dropout layer is gone)

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])

In [102]:
torch.manual_seed(123)
dropout(attn_weights)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6803, 0.6590, 0.6607, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4947, 0.4953, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3949, 0.0000, 0.4025, 0.0000, 0.0000],
        [0.0000, 0.3269, 0.3274, 0.3365, 0.3447, 0.0000]],
       grad_fn=<MulBackward0>)

### Attention Class

In [115]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super(CausalAttention, self).__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(p=dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(
            1, 2
        )  # transpose inside the keys, do not touch the batches (0->batch)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / self.d_out**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vec = attn_weights @ values
        return context_vec

In [116]:
batch = torch.stack((inputs, inputs), dim=0)
batch

tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])

In [117]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print(f"context_vecs.shape: {context_vecs.shape}")

context_vecs.shape: torch.Size([2, 6, 2])


In [118]:
context_length

6

In [132]:
longer_ctx = torch.concat((inputs, inputs))
ca(torch.stack([longer_ctx, longer_ctx]))

RuntimeError: The size of tensor a (12) must match the size of tensor b (6) at non-singleton dimension 2

In [ ]:
longer_ctx = torch.concat((inputs, inputs))
ca(torch.stack([longer_ctx, longer_ctx])[:, :5, ...])

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981]]], grad_fn=<UnsafeViewBackward0>)

### Multi-head, Naive way

In [137]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super(MultiHeadAttentionWrapper, self).__init__()
        self.heads = nn.ModuleList(
            [
                CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
                for _ in range(num_heads)
            ]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [138]:
torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


In [140]:
# Exercise 3.2
torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3, 1
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.5740,  0.2216],
         [-0.7320,  0.0155],
         [-0.7774, -0.0546],
         [-0.6979, -0.0817],
         [-0.6538, -0.0957],
         [-0.6424, -0.1065]],

        [[-0.5740,  0.2216],
         [-0.7320,  0.0155],
         [-0.7774, -0.0546],
         [-0.6979, -0.0817],
         [-0.6538, -0.0957],
         [-0.6424, -0.1065]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


In [163]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super(MultiHeadAttention, self).__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = self.d_out // self.num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(p=dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(
            b, num_tokens, self.num_heads, self.head_dim
        )  # Now keys have num_heads blocks inside
        # e.g. before we had only 1 head, so it was (batch, num_tokens, d_out). If num_heads was 1, it was gonna be (batch, num_tokens, 1, d_out)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # brought them into batch, head, num_tokens, d_out shape
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)
        return context_vec

In [164]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


In [165]:
# Exercise 3.3
mha = MultiHeadAttention(
    d_in=768, d_out=768, context_length=1024, dropout=0.0, num_heads=12
)

In [170]:
batch = torch.rand([2, 6, 768])

In [172]:
mha(batch).shape

torch.Size([2, 6, 768])

In [ ]:
%%writefile mha.py
# From: Chapter 3
import torch
from torch import nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super(MultiHeadAttention, self).__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"
        
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = self.d_out // self.num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(p=dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) # Now keys have num_heads blocks inside
        # e.g. before we had only 1 head, so it was (batch, num_tokens, d_out). If num_heads was 1, it was gonna be (batch, num_tokens, 1, d_out)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # brought them into batch, head, num_tokens, d_out shape
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2) 

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)
        return context_vec

Overwriting mha.py
